# Möbius homology calculator — Colab version

This notebook is self-contained and should run directly in Google Colab.

It computes the chain complex for a perfect matching graph using either the algebra $V_{n,0}=\mathbb C[x,y]/(x^n, y^3-xy)$ or $V_{n,1}=\mathbb C[x,y]/(x^n-1, y^3-xy)$.

The implementation tracks the actual smoothing circles/components, uses genuine cube edges only, applies $\mu$, $\Delta$, and $m$ to the correct tensor factors, and uses complex number evaluation via `sympy`, together with a numerical tolerance of 1e^-9. To compute the ranks of differential matrices, an SVD-based rank calculation is used via `numpy`.

Run the cells from top to bottom. The final cells contain editable examples.


In [1]:
# Ensure both SymPy and NumPy are installed and print their versions
try:
    import sympy as sp
    import numpy as np
    print('SymPy version:', sp.__version__)
    print('NumPy version:', np.__version__)
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sympy', 'numpy'])
    import sympy as sp
    import numpy as np
    print('SymPy version:', sp.__version__)
    print('NumPy version:', np.__version__)

SymPy version: 1.14.0
NumPy version: 2.1.3


## Core code

You usually do not need to edit this cell. Edit the examples below instead.

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations, product
from typing import Dict, Iterable, List, Sequence, Tuple, Union, Optional

import numpy as np
import sympy as sp
from sympy import oo

Endpoint = Tuple[int, int]  # (vertex, side), side is -1 or +1
Edge = Tuple[Endpoint, Endpoint]
State = Tuple[int, ...]  # tuple of 0s and 1s
BasisElt = Tuple[int, int]  # a^i b^j with 0 <= i < n and j in {0,1,2}
TensorElt = Tuple[int, ...]


def _canonical_edge(u: Endpoint, v: Endpoint) -> Edge:
    """Orders edge endpoints consistently."""
    return (u, v) if u <= v else (v, u)


def _binary_states(num_bits: int, weight: int) -> List[State]:
    """Generates all binary states with a given number of 1s (weight)."""
    out: List[State] = []
    for ones in combinations(range(num_bits), weight):
        arr = [0] * num_bits
        for i in ones:
            arr[i] = 1
        out.append(tuple(arr))
    return out


def _cube_sign(source: State, changed_position: int) -> int:
    """Standard cube sign: (-1)^(number of 1s before the changed coordinate)."""
    return -1 if sum(source[:changed_position]) % 2 else 1


@dataclass(frozen=True)
class SmoothingData:
    state: State
    circles: Tuple[frozenset[Endpoint], ...]

    @property
    def k(self) -> int:
        return len(self.circles)


class MobiusAlgebra:
    """The algebra C[x,y]/(x^n-xi, y^3-x*y) with the Möbius Frobenius structure."""

    def __init__(self, n: int = 1, xi: bool = True):
        if n < 1:
            raise ValueError("n must be positive")
        self.n = int(n)
        self.xi = bool(xi)
        self.basis: List[BasisElt] = [(i, j) for i in range(self.n) for j in range(3)]
        self.index: Dict[BasisElt, int] = {b: i for i, b in enumerate(self.basis)}
        self.dim = len(self.basis)
        self._pairing_inv: Optional[sp.Matrix] = None
        self._delta_one: Optional[Dict[Tuple[int, int], complex]] = None

    def degree_of_index(self, idx: int) -> int:
        """Calculates the internal degree of a basis element a^i b^j."""
        i, j = self.basis[idx]
        if self.n % 2 == 0:
            return (2 * i + j) % self.n
        else:
            if self.xi:
                return j % 2
            else:
                return (2 * i + j) % (self.n + 1)

    def tensor_degree(self, tensor: TensorElt) -> int:
        """Calculates the total internal degree of a tensor product of basis elements."""
        if self.n % 2 == 0:
            return sum(self.degree_of_index(i) for i in tensor) % self.n
        else:
            if self.xi:
                return sum(self.degree_of_index(i) for i in tensor) % 2
            else:
                return sum(self.degree_of_index(i) for i in tensor) % (self.n + 1)

    def multiply_basis(self, p: int, q: int) -> Dict[int, complex]:
        """Product of two basis elements, returned sparsely (as dict)."""
        ai, bj = self.basis[p]
        ak, bl = self.basis[q]
        a_exp = ai + ak
        b_exp = bj + bl
        while b_exp >= 3:
            a_exp += 1
            b_exp -= 2
        if self.xi:
            return {self.index[(a_exp % self.n, b_exp)]: complex(1)}
        else:
            if a_exp >= self.n:
                return {}
            return {self.index[(a_exp, b_exp)]: complex(1)}

    @property
    def mobius_element(self) -> Dict[int, complex]:
        """The Möbius element m as a sum of basis elements."""
        if self.n % 2 == 0:
            idx = self.index[((self.n - 2) // 2, 2)]
            idx_0 = self.index[(self.n // 2, 0)]
            return {
                idx: complex(1 / sp.sqrt(3) * (sp.sqrt(2) - sp.I)),
                idx_0: complex(1 / sp.sqrt(3) * sp.I),
            }
        else:
            if self.n == 1:
                if self.xi:
                    idx = self.index[((self.n - 1) // 2, 2)]
                    idx_0 = self.index[(0, 0)]
                    return {
                        idx: complex(1 / sp.sqrt(3) * (sp.sqrt(2) - sp.I)),
                        idx_0: complex(1 / sp.sqrt(3) * sp.I),
                    }
                else:
                    idx = self.index[((self.n - 1) // 2, 2)]
                    return {idx: complex(1 / sp.sqrt(3) * (sp.sqrt(2) - sp.I))}
            else:
                idx = self.index[((self.n - 1) // 2, 2)]
                idx_0 = self.index[((self.n + 1) // 2, 0)]
                return {
                    idx: complex(1 / sp.sqrt(3) * (sp.sqrt(2) - sp.I)),
                    idx_0: complex(1 / sp.sqrt(3) * sp.I),
                }

    def mobius_map_on_basis(self, p: int) -> Dict[int, complex]:
        """Product of the Mobius element with a basis element, returned sparsely."""
        out: Dict[int, complex] = {}
        for mob_idx, mob_coeff in self.mobius_element.items():
            for prod_idx, prod_coeff in self.multiply_basis(mob_idx, p).items():
                out[prod_idx] = out.get(prod_idx, complex(0)) + mob_coeff * prod_coeff
        return {k: v for k, v in out.items() if v}

    def epsilon_on_basis(self, p: int) -> complex:
        """The trace map epsilon(e_p)."""
        return complex(3 * self.n) if p == self.dim - 1 else complex(0)

    def pairing_matrix(self) -> sp.Matrix:
        """Computes the pairing matrix (e_i, e_j) = epsilon(e_i * e_j)."""
        M = sp.zeros(self.dim, self.dim)
        for i in range(self.dim):
            for j in range(self.dim):
                s = complex(0)
                for k, c in self.multiply_basis(i, j).items():
                    s += c * self.epsilon_on_basis(k)
                M[i, j] = sp.sympify(s)
        return M

    def pairing_inverse(self) -> sp.Matrix:
        """Returns the inverse of the pairing matrix, caching the result."""
        if self._pairing_inv is None:
            self._pairing_inv = self.pairing_matrix().inv()
        return self._pairing_inv

    def delta_one(self) -> Dict[Tuple[int, int], complex]:
        if self.n % 2 == 0 or (self.n == 1 and self.xi):
            if self._delta_one is None:
                inv = self.pairing_inverse()
                d: Dict[Tuple[int, int], complex] = {}
                for i in range(self.dim):
                    for j in range(self.dim):
                        val = inv[i, j]
                        if val:
                            d[(i, j)] = complex(val)
                self._delta_one = d
            return self._delta_one
        elif self.n != 1:
            if self._delta_one is None:
                d: Dict[Tuple[int, int], complex] = {}
                for e1 in self.basis:
                    for e2 in self.basis:
                        i = self.index[e1]
                        j = self.index[e2]
                        val_1 = complex(-1 / (3 * self.n))
                        val_2 = complex(1 / (3 * self.n))
                        if self.xi:
                            if (e1[0] + e2[0]) % self.n == 1 and e1[1] + e2[1] == 0:
                                d[(i, j)] = val_1
                            elif (e1[0] + e2[0]) % self.n == 0 and e1[1] + e2[1] == 2:
                                d[(i, j)] = val_2
                        else:
                            if e1[0] + e2[0] == self.n + 1 and e1[1] + e2[1] == 0:
                                d[(i, j)] = val_1
                            elif e1[0] + e2[0] == self.n and e1[1] + e2[1] == 2:
                                d[(i, j)] = val_2
                self._delta_one = d
            return self._delta_one
        else:
            self._delta_one = {}
            return self._delta_one

    def delta_on_basis(self, p: int) -> Dict[Tuple[int, int], complex]:
        """Delta(e_p) = (e_p tensor 1) Delta(1)."""
        out: Dict[Tuple[int, int], complex] = {}
        for (i, j), c in self.delta_one().items():
            for k, ck in self.multiply_basis(p, i).items():
                prod_coeff = c * ck
                out[(k, j)] = out.get((k, j), complex(0)) + prod_coeff
        return {k: v for k, v in out.items() if v}


class MobiusHomology:
    def __init__(
        self,
        cycles: Sequence[Sequence[Union[int, List[int]]]],
        matching: Optional[Sequence[Sequence[int]]] = None,
        n: int = 1,
        xi: bool = True,
        tolerance: float = 1e-9,
    ):
        self.cycles = cycles
        if matching is not None:
            pairs = self.cyc_2_pair(matching)
            self.pairs: List[List[int]] = [list(map(int, p)) for p in pairs]
            self.matching: List[Tuple[int, int]] = [tuple(map(int, m)) for m in matching]
        else:
            pairs, computed_matching = self.graph_2_blowup()
            self.pairs = [list(map(int, p)) for p in pairs]
            self.matching = [tuple(map(int, m)) for m in computed_matching]
        self.num_matchings = len(self.matching)
        self.alg = MobiusAlgebra(n=n, xi=xi)
        self._base_edges = self._non_matching_edges()
        self._smoothing_cache: Dict[State, SmoothingData] = {}
        self._tensor_basis_cache: Dict[Tuple[int, Optional[int]], List[TensorElt]] = {}
        self.tolerance = tolerance

    def cyc_2_pair(self, matching: Sequence[Sequence[int]]) -> List[List[int]]:
        matching_sort = []
        for m in matching:
            matching_sort.append(sorted(m))
        matching = matching_sort

        pairs: List[List[int]] = [[] for _ in range(len(self.cycles))]

        for m in matching:
            s = self.cycles[m[0]]
            q = len(s)
            r = q - 1
            if q == 3:
                for i in range(q):
                    if s[i] == m[1]:
                        pairs[m[0]] = [int(s[(i - 1) % q]), int(s[(i + 1) % q])]
            elif r == 3:
                swap = s[3]
                s_sliced = s[:-1]
                for i in range(r):
                    if s_sliced[i] == m[1]:
                        if swap == 0:
                            pairs[m[0]] = [int(s_sliced[(i - 1) % r]), int(s_sliced[(i + 1) % r]), 0]
                        else:
                            pairs[m[0]] = [int(s_sliced[(i - 1) % r]), int(s_sliced[(i + 1) % r]), 1]

            t = self.cycles[m[1]]
            v = len(t)
            w = v - 1
            if v == 3:
                for i in range(v):
                    if t[i] == m[0]:
                        pairs[m[1]] = [int(t[(i + 1) % v]), int(t[(i - 1) % v])]
            elif w == 3:
                swap = t[3]
                t_sliced = t[:-1]
                for i in range(w):
                    if t_sliced[i] == m[0]:
                        if swap == 0:
                            pairs[m[1]] = [int(t_sliced[(i + 1) % w]), int(t_sliced[(i - 1) % w]), 0]
                        else:
                            pairs[m[1]] = [int(t_sliced[(i + 1) % w]), int(t_sliced[(i - 1) % w]), 1]
        return pairs

    def flatten(self, matrix: List[List[int]]) -> List[int]:
        flat_list = []
        for row in matrix:
            for element in row:
                flat_list.append(element)
        return flat_list

    def graph_2_blowup(self) -> Tuple[List[List[int]], List[List[int]]]:
        group_index = 0
        groups: List[List[int]] = []
        for cycle in self.cycles:
            groups.append(list(range(group_index, group_index + len(cycle))))
            group_index += len(cycle)

        matching: List[List[int]] = []
        used_elements = set()

        for i in range(len(self.cycles)):
            for j in range(len(self.cycles[i])):
                item = self.cycles[i][j]
                if isinstance(item, list):
                    ind, part = item
                    if (
                        groups[i][j] not in used_elements
                        and groups[ind][part] not in used_elements
                    ):
                        matching.append(sorted([groups[i][j], groups[ind][part]]))
                        used_elements.add(groups[i][j])
                        used_elements.add(groups[ind][part])
                else:
                    ind = int(item)
                    for k in range(len(self.cycles[ind])):
                        candidate = self.cycles[ind][k]
                        if not isinstance(candidate, list) and candidate == i:
                            if (
                                groups[i][j] not in used_elements
                                and groups[ind][k] not in used_elements
                            ):
                                matching.append(
                                    sorted([groups[i][j], groups[ind][k]])
                                )
                                used_elements.add(groups[i][j])
                                used_elements.add(groups[ind][k])
                                break

        pairs: List[List[int]] = [[] for _ in range(len(self.flatten(groups)))]
        for i in range(len(self.flatten(groups))):
            for j in range(len(matching)):
                for k in range(len(groups)):
                    if i in groups[k]:
                        q = len(groups[k])
                        if matching[j][0] == i:
                            pairs[i] = [
                                groups[k][0] + (i - 1) % q,
                                groups[k][0] + (i + 1) % q,
                            ]
                        elif matching[j][1] == i:
                            pairs[i] = [
                                groups[k][0] + (i + 1) % q,
                                groups[k][0] + (i - 1) % q,
                            ]
        return pairs, matching

    def _non_matching_edges(self) -> List[Edge]:
        raw: List[Edge] = []
        pairs = self.pairs
        for i in range(len(pairs)):
            left, right = pairs[i][0], pairs[i][1]
            if len(pairs[i]) == 2 or pairs[i][2] == 0:
                if left != right:
                    if pairs[left][0] == i:
                        raw.append(_canonical_edge((i, -1), (left, -1)))
                    elif pairs[left][1] == i:
                        raw.append(_canonical_edge((i, -1), (left, 1)))
                    if pairs[right][1] == i:
                        raw.append(_canonical_edge((i, 1), (right, 1)))
                    elif pairs[right][0] == i:
                        raw.append(_canonical_edge((i, 1), (right, -1)))
                else:
                    if left == i:
                        raw.append(_canonical_edge((i, -1), (left, 1)))
                        raw.append(_canonical_edge((i, 1), (left, -1)))
                    else:
                        raw.append(_canonical_edge((i, -1), (left, -1)))
                        raw.append(_canonical_edge((i, 1), (left, 1)))
            else:
                raw.append(_canonical_edge((i, -1), (left, 1)))
                raw.append(_canonical_edge((i, 1), (left, -1)))

        seen = set()
        out = []
        for e in raw:
            if e not in seen:
                seen.add(e)
                out.append(e)
        return out

    def smoothing_edges(self, state: State) -> List[Edge]:
        if len(state) != self.num_matchings:
            raise ValueError("state length does not match number of matching edges")
        edges = list(self._base_edges)
        for bit, (u, v) in zip(state, self.matching):
            if bit == 0:
                edges.append(_canonical_edge((u, 1), (v, 1)))
                edges.append(_canonical_edge((u, -1), (v, -1)))
            elif bit == 1:
                edges.append(_canonical_edge((u, 1), (v, -1)))
                edges.append(_canonical_edge((u, -1), (v, 1)))
            else:
                raise ValueError("states must be 0/1")
        return edges

    def smoothing_data(self, state: State) -> SmoothingData:
        state = tuple(state)
        if state in self._smoothing_cache:
            return self._smoothing_cache[state]

        adj: Dict[Endpoint, List[Endpoint]] = {}
        for u, v in self.smoothing_edges(state):
            adj.setdefault(u, []).append(v)
            adj.setdefault(v, []).append(u)

        bad = {u: vs for u, vs in adj.items() if len(vs) != 2}
        if bad:
            raise ValueError(
                f"smoothing {state} is not a disjoint union of circles; bad endpoints: {bad}"
            )

        seen = set()
        circles: List[frozenset[Endpoint]] = []
        for start in sorted(adj):
            if start in seen:
                continue
            comp = set()
            stack = [start]
            seen.add(start)
            while stack:
                u = stack.pop()
                comp.add(u)
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            circles.append(frozenset(comp))
        circles.sort(key=lambda c: sorted(c))
        data = SmoothingData(state=state, circles=tuple(circles))
        self._smoothing_cache[state] = data
        return data

    def states_by_homological_degree(self) -> List[List[State]]:
        return [
            _binary_states(self.num_matchings, i)
            for i in range(self.num_matchings + 1)
        ]

    def state_circle_counts(self) -> List[List[int]]:
        return [
            [self.smoothing_data(s).k for s in states]
            for states in self.states_by_homological_degree()
        ]

    def cube_edges(self, degree: int) -> List[Tuple[State, State, int, int]]:
        out = []
        for s in _binary_states(self.num_matchings, degree):
            for pos, bit in enumerate(s):
                if bit == 0:
                    t = list(s)
                    t[pos] = 1
                    t_tuple = tuple(t)
                    out.append((s, t_tuple, pos, _cube_sign(s, pos)))
        return out

    def tensor_basis(
        self, k: int, internal_degree: Optional[int] = None
    ) -> List[TensorElt]:
        key = (k, internal_degree)
        if key in self._tensor_basis_cache:
            return self._tensor_basis_cache[key]
        all_tuples = list(product(range(self.alg.dim), repeat=k))
        if internal_degree is not None:
            if self.alg.n % 2 == 1 and self.alg.xi:
                all_tuples = [
                    v
                    for v in all_tuples
                    if self.alg.tensor_degree(v) == internal_degree % 2
                ]
            elif self.alg.n % 2 == 1 and not self.alg.xi:
                all_tuples = [
                    v
                    for v in all_tuples
                    if self.alg.tensor_degree(v) == internal_degree % (self.alg.n + 1)
                ]
            else:
                all_tuples = [
                    v
                    for v in all_tuples
                    if self.alg.tensor_degree(v) == internal_degree % self.alg.n
                ]
        self._tensor_basis_cache[key] = all_tuples
        return all_tuples

    def chain_basis(
        self, degree: int, internal_degree: Optional[int] = None
    ) -> List[Tuple[State, TensorElt]]:
        out: List[Tuple[State, TensorElt]] = []
        for state in _binary_states(self.num_matchings, degree):
            k = self.smoothing_data(state).k
            for ten in self.tensor_basis(k, internal_degree):
                out.append((state, ten))
        return out

    def _component_indices_touching(
        self, data: SmoothingData, endpoints: Iterable[Endpoint]
    ) -> List[int]:
        endpoints_set = set(endpoints)
        return [i for i, c in enumerate(data.circles) if c & endpoints_set]

    def _edge_map_on_tensor(
        self, source: State, target: State, changed_pos: int, x: TensorElt
    ) -> Dict[TensorElt, complex]:
        sd = self.smoothing_data(source)
        td = self.smoothing_data(target)
        u, v = self.matching[changed_pos]
        touched = [(u, -1), (u, 1), (v, -1), (v, 1)]
        src_touch = self._component_indices_touching(sd, touched)
        tgt_touch = self._component_indices_touching(td, touched)

        source_comp_to_target: Dict[int, int] = {}
        for i, c in enumerate(sd.circles):
            for j, d in enumerate(td.circles):
                if c == d:
                    source_comp_to_target[i] = j

        out: Dict[TensorElt, complex] = {}

        if sd.k == td.k + 1:
            if len(src_touch) != 2 or len(tgt_touch) != 1:
                raise ValueError(
                    f"unexpected merge data {source}->{target}: {src_touch}, {tgt_touch}"
                )
            a, b = src_touch
            tgt = tgt_touch[0]
            local = self.alg.multiply_basis(x[a], x[b])
            for prod_idx, coeff in local.items():
                y = [None] * td.k
                y[tgt] = prod_idx
                for i in range(sd.k):
                    if i not in (a, b):
                        y[source_comp_to_target[i]] = x[i]
                out[tuple(y)] = out.get(tuple(y), complex(0)) + coeff

        elif sd.k + 1 == td.k:
            if len(src_touch) != 1 or len(tgt_touch) != 2:
                raise ValueError(
                    f"unexpected split data {source}->{target}: {src_touch}, {tgt_touch}"
                )
            src = src_touch[0]
            t0, t1 = tgt_touch
            local = self.alg.delta_on_basis(x[src])
            for (left, right), coeff in local.items():
                y = [None] * td.k
                y[t0] = left
                y[t1] = right
                for i in range(sd.k):
                    if i != src:
                        y[source_comp_to_target[i]] = x[i]
                out[tuple(y)] = out.get(tuple(y), complex(0)) + coeff

        elif sd.k == td.k:
            if len(src_touch) != 1 or len(tgt_touch) != 1:
                raise ValueError(
                    f"unexpected Mobius data {source}->{target}: {src_touch}, {tgt_touch}"
                )
            src = src_touch[0]
            tgt = tgt_touch[0]
            local = self.alg.mobius_map_on_basis(x[src])
            for mob_idx, coeff in local.items():
                y = [None] * td.k
                y[tgt] = mob_idx
                for i in range(sd.k):
                    if i != src:
                        y[source_comp_to_target[i]] = x[i]
                out[tuple(y)] = out.get(tuple(y), complex(0)) + coeff
        else:
            raise ValueError(
                f"circle count changed by more than one: {source}->{target}"
            )

        clean: Dict[TensorElt, complex] = {}
        for y, c in out.items():
            if any(z is None for z in y):
                raise RuntimeError(
                    f"internal error: incomplete output tensor {y} for {source}->{target}"
                )
            clean[y] = clean.get(y, complex(0)) + c
        return {k: v for k, v in clean.items() if v}

    def differential_matrix(
        self, degree: int, internal_degree: Optional[int] = None
    ) -> sp.Matrix:
        if degree < 0 or degree >= self.num_matchings:
            return sp.zeros(0, 0)
        src_basis = self.chain_basis(degree, internal_degree)
        tgt_basis = self.chain_basis(degree + 1, internal_degree)
        src_index = {b: i for i, b in enumerate(src_basis)}
        tgt_index = {b: i for i, b in enumerate(tgt_basis)}
        D = sp.MutableSparseMatrix(len(tgt_basis), len(src_basis), {})

        for source, target, changed_pos, sign in self.cube_edges(degree):
            for x in self.tensor_basis(self.smoothing_data(source).k, internal_degree):
                col = src_index[(source, x)]
                for y, coeff in self._edge_map_on_tensor(
                    source, target, changed_pos, x
                ).items():
                    if self.alg.n % 2 == 1 and self.alg.xi:
                        if (
                            internal_degree is not None
                            and self.alg.tensor_degree(y) != internal_degree % 2
                        ):
                            raise RuntimeError(
                                "local map did not preserve internal degree"
                            )
                    elif self.alg.n % 2 == 1 and not self.alg.xi:
                        if (
                            internal_degree is not None
                            and self.alg.tensor_degree(y)
                            != internal_degree % (self.alg.n + 1)
                        ):
                            raise RuntimeError(
                                "local map did not preserve internal degree"
                            )
                    else:
                        if (
                            internal_degree is not None
                            and self.alg.tensor_degree(y) != internal_degree % self.alg.n
                        ):
                            raise RuntimeError(
                                "local map did not preserve internal degree"
                            )
                    row = tgt_index[(target, y)]
                    val = complex(sign) * coeff
                    D[row, col] += sp.sympify(val)
        return sp.Matrix(D)

    def homology_dimensions(self, internal_degree: Optional[int] = None) -> List[int]:
        if internal_degree is None:
            all_filtered_dims_dict = (
                self.homology_dimensions_by_internal_degree()
            )
            if not all_filtered_dims_dict:
                return [0] * (self.num_matchings + 1)

            num_homological_degrees = self.num_matchings + 1
            summed_dims = [0] * num_homological_degrees
            for ideg_dims in all_filtered_dims_dict.values():
                for i in range(num_homological_degrees):
                    summed_dims[i] += ideg_dims[i]
            return summed_dims

        dims = []
        prev_rank = 0
        for i in range(self.num_matchings + 1):
            chain_dim = len(self.chain_basis(i, internal_degree))
            if i < self.num_matchings:
                D_i = self.differential_matrix(i, internal_degree)
                D_i_np = np.array(D_i.tolist(), dtype=complex)
                s = np.linalg.svd(D_i_np, compute_uv=False)
                rank_next = int(np.sum(s > self.tolerance))
            else:
                rank_next = 0
            dims.append(int(chain_dim - rank_next - prev_rank))
            prev_rank = rank_next
        return dims

    def homology_dimensions_by_internal_degree(self) -> Dict[int, List[int]]:
        if self.alg.n % 2 == 1 and self.alg.xi:
            return {d: self.homology_dimensions(d) for d in range(2)}
        elif self.alg.n % 2 == 1 and not self.alg.xi:
            return {d: self.homology_dimensions(d) for d in range(self.alg.n + 1)}
        else:
            return {d: self.homology_dimensions(d) for d in range(self.alg.n)}

    def check_d_squared(self, internal_degree: Optional[int] = None) -> bool:
        tolerance = self.tolerance
        for i in range(self.num_matchings - 1):
            D_i = self.differential_matrix(i, internal_degree)
            D_i_plus_1 = self.differential_matrix(i + 1, internal_degree)
            product_matrix = D_i_plus_1 * D_i
            current_norm = product_matrix.norm(oo)
            if current_norm > tolerance:
                print(
                    f"  d^2 != 0 detected at degree i={i}. Norm {current_norm} > tolerance {tolerance}"
                )
                return False
        return True

    def poincare_polynomial(
        self, variable_t: str = "t", variable_q: str = "q", variable_b: str = "b"
    ) -> str:
        t, q, b = sp.symbols(f"{variable_t} {variable_q} {variable_b}")
        poly = 0
        for d, hs in self.homology_dimensions_by_internal_degree().items():
            for i, dim in enumerate(hs):
                if self.alg.n % 2 == 0 or self.alg.n == 1 or not self.alg.xi:
                    if d % 2 == 0:
                        poly += dim * (q**d) * (t**i)
                    else:
                        poly += dim * (q**d) * b * (t**i)
                else:
                    poly += dim * (b**d) * (t**i)
        return str(sp.expand(poly))

## Test example

This should print circle counts `[[2], [1]]`, verify `d^2=0`, and give homology dimensions `[6, 0]`, and  Poincare polynomial `3*b*q + 3`.

In [13]:
example_1_cycles = [[1, 1, 1, 0], [0, 0, 0, 0]]
example_1_matching = [[0, 1]]

MH = MobiusHomology(example_1_cycles, example_1_matching, n=1, xi=True)
print("circle counts:", MH.state_circle_counts())
print("d^2=0:", MH.check_d_squared())
print("homology dimensions:", MH.homology_dimensions())
print("n=1, xi=1 Poincare polynomial:", MH.poincare_polynomial())

circle counts: [[2], [1]]
d^2=0: True
homology dimensions: [6, 0]
n=1, xi=1 Poincare polynomial: 3*b*q + 3


## Another example

The following example is a variation of the first example in the core code, but now instead of specifying a perfect matching, the blowup is selected.

In [12]:
cycles = [[[1, 2], [1, 1], [1, 0]], [[0, 2], [0, 1], [0, 0]]]

MH = MobiusHomology(cycles=cycles, n=1, xi=True)
print('circle counts:', MH.state_circle_counts())
print('d^2 = 0:', MH.check_d_squared())
print('homology dimensions:', MH.homology_dimensions())
print('n=1, xi=1 Filtered Poincare polynomial:', MH.poincare_polynomial())

circle counts: [[3], [2, 2, 2], [1, 1, 1], [1]]
d^2 = 0: True
homology dimensions: [6, 0, 0, 0]
n=1, xi=1 Filtered Poincare polynomial: 3*b*q + 3


## Edit this cell for your own graph

Input format is given by:

```python
# for ribbon graph blowups
cycles[i] = [connected_vertex_1, connected_vertex_2, ..., [connected_vertex_k, vertex_index_k], ...]  

# perfect matching specified for trivalent graphs
cycles[i] = [connected_vertex_1, connected_vertex_2, connected_vertex_3, crossing_data]
matching = [[u, v], ...]
```

Set `n=1` and `xi=True` for the algebra $\mathbb C[y]/(y^3-y)$ or `n=1` and `xi=False` for the algebra $\mathbb C[y]/(y^3)$.

In [3]:
# Replace with your ribbon graph data to obtain the blowup
# cycles = [[[1, 2], [1, 1], [1, 0]], [[0, 2], [0, 1], [0, 0]]]

# Alternatively, you can input perfect matching data
cycles = [[1, 1, 1, 0], [0, 0, 0, 0]]
matching = [[0, 1]]

# Choice of parameter n
n = 1

# Boolean input: True for xi = 1 or False for xi = 0
xi = True

try:
    matching
except NameError:
    MH = MobiusHomology(cycles=cycles, n=n, xi=xi)
else:
    MH = MobiusHomology(cycles=cycles, matching=matching, n=n, xi=xi)
print('circle counts by homological degree:', MH.state_circle_counts())
print('d^2 = 0:', MH.check_d_squared())
print('homology dimensions by homological degree:', MH.homology_dimensions())
print('homology dimensions by internal degree:', MH.homology_dimensions_by_internal_degree())
if xi:
  print(f'n = {n}, xi = 1 Poincare polynomial:', MH.poincare_polynomial())
else:
  print(f'n = {n}, xi = 0 Poincare polynomial:', MH.poincare_polynomial())

circle counts by homological degree: [[2], [1]]
d^2 = 0: True
homology dimensions by homological degree: [6, 0]
homology dimensions by internal degree: {0: [3, 0], 1: [3, 0]}
n = 1, xi = 1 Poincare polynomial: 3*b*q + 3


## Optional: inspect a differential matrix

For example, `degree=0` prints the matrix of $d^0:C^0	\rightarrow C^1$.

In [9]:
degree = 0
D = MH.differential_matrix(degree)
print('shape:', D.shape)
D



shape: (18, 9)


Matrix([
[                                      0,                 0,                 0,                     0.577350269189626*I,                 0,                 0,                                       0,                 0,                 0],
[                                      0,                 0,                 0,                                       0, 0.816496580927726,                 0,                                       0,                 0,                 0],
[                                      0,                 0,                 0,                                       0,                 0, 0.816496580927726, 0.816496580927726 - 0.577350269189626*I,                 0,                 0],
[                                      0,                 0,                 0,                                       0,                 0,                 0,                     0.577350269189626*I,                 0,                 0],
[                                  

## Shortcuts for graph families

The following code produces the `cycles` data for cycle graphs $C_m$, path graphs $P_m$, complete graphs $K_m$, and complete bipartite graphs $K_{m, p}$. You do not need to edit this cell.

In [15]:
from typing import List, Union, Tuple


# Cycle graph converter
def cyc2cyc(m: int) -> List[List[Union[int, List[int], Tuple[int, int]]]]:
    if m < 1:
        raise ValueError("m must be positive")

    if m == 1:
        return [[[0, 1], [0, 0]]]
    elif m == 2:
        return [[[1, 1], [1, 0]], [[0, 1], [0, 0]]]
    else:
        cycles: List[List[int]] = [[] for _ in range(m)]
        for i in range(m):
            cycles[i] = [(i + 1) % m, (i - 1) % m]
        return cycles


# Path graph converter
def path2cyc(m: int) -> List[List[int]]:
    if m < 2:
        raise ValueError("m must be a positive integer > 1")

    cycles: List[List[int]] = [[] for _ in range(m)]
    cycles[0] = [1]
    cycles[-1] = [m - 2]

    for i in range(1, m - 1):
        cycles[i] = [(i + 1) % m, (i - 1) % m]

    return cycles


# Complete graph converter
def complete2cyc(m: int) -> List[List[int]]:
    if m < 1:
        raise ValueError("m must be an integer > 1")

    cycles: List[List[int]] = [[] for _ in range(m)]
    for i in range(m):
        for j in range(1, m):
            cycles[i].append((i + j) % m)

    return cycles


# Complete bipartite graph converter
def compbipartite2cyc(m: int, p: int) -> List[List[int]]:
    if m < 1 or p < 1:
        raise ValueError("m and p must be positive integers")

    cycles: List[List[int]] = [[] for _ in range(m + p)]
    for i in range(m):
        cycles[i] = [val for val in range(m, m + p)]

    for i in range(m, m + p):
        cycle = [val for val in range(m)]
        cycles[i] = cycle[::-1]

    return cycles

You can uncomment one of the options below to run the main code on your choice of graph family indexed by a given integer(s).

In [16]:
# Choice of parameter n
n = 1

# Boolean input: True for xi = 1 or False for xi = 0
xi = True

# Cycle graphs
m = 1
cycles = cyc2cyc(m)

# Path graphs
# m = 2
# cycles = path2cyc(m)

# Complete graphs
# m = 2
# cycles = complete2cyc(m)

# Complete bipartite graphs
# m = 1
# p = 1
# cycles = compbipartite2cyc(m, p)

MH = MobiusHomology(cycles=cycles, n=n, xi=xi)
print('circle counts by homological degree:', MH.state_circle_counts())
print('d^2 = 0:', MH.check_d_squared())
print('homology dimensions by homological degree:', MH.homology_dimensions())
print('homology dimensions by internal degree:', MH.homology_dimensions_by_internal_degree())
if xi:
  print(f'n = {n}, xi = 1 Poincare polynomial:', MH.poincare_polynomial())
else:
  print(f'n = {n}, xi = 0 Poincare polynomial:', MH.poincare_polynomial())

circle counts by homological degree: [[2], [1]]
d^2 = 0: True
homology dimensions by homological degree: [6, 0]
homology dimensions by internal degree: {0: [3, 0], 1: [3, 0]}
n = 1, xi = 1 Poincare polynomial: 3*b*q + 3


## Number of face colorings

The following code calculates the number of $(3n - |A|)$-face colorings of any ribbon graph via the blowup using the degree zero differential of the $\xi=1$ Möbius homology, where $A\subseteq \{0, \ldots, n-1\}$. You do not need to edit this cell.


In [7]:
from typing import List, Union, Dict, Set
import numpy as np


def get_face_color(cycles: List[List[Union[int, List[int]]]], n: int, A: List[int]) -> int:
    MH = MobiusHomology(cycles=cycles, n=n, xi=True)

    degree: int = 0
    D = MH.differential_matrix(degree)

    # Convert SymPy matrix to a dense NumPy float/complex matrix for fast numerical nullspace computation
    D_np = np.array(D.tolist(), dtype=complex)

    # Compute SVD to find the numerical nullspace
    u, s, vh = np.linalg.svd(D_np)

    # Identify singular values close to zero (nullspace basis vectors)
    num_cols = D_np.shape[1]
    num_singular_values = len(s)

    nullspace_vectors = []
    for i in range(num_singular_values):
        if s[i] < MH.tolerance:
            nullspace_vectors.append(vh[i, :].conj())
    for i in range(num_singular_values, num_cols):
        nullspace_vectors.append(vh[i, :].conj())

    # If no nullspace exists, return 0
    if not nullspace_vectors:
        return 0

    # Convert nullspace vectors directly to dense NumPy column arrays for numerical operations
    nullspace_vectors_np = [v.reshape(-1, 1) for v in nullspace_vectors]
    null_len = len(nullspace_vectors_np)

    omega = np.exp(2 * np.pi * 1j / n)

    def p_k(k: int) -> Dict[int, complex]:
        pk_coeffs: Dict[int, complex] = {}
        for i_exp in range(MH.alg.n):
            basis_elt = (i_exp, 0)
            basis_index = MH.alg.index[basis_elt]
            coefficient = omega ** (-i_exp * k)
            pk_coeffs[basis_index] = 1 / MH.alg.n * coefficient
        return pk_coeffs

    def E_k(k: int, v: int) -> Dict[int, complex]:
        pk_coeffs = p_k(k)
        if v == 0:
            mult_term_coeffs = {
                MH.alg.index[(0, 0)]: complex(1),
                MH.alg.index[(MH.alg.n - 1, 2)]: complex(-1),
            }
        elif v == 1:
            mult_term_coeffs = {
                MH.alg.index[(MH.alg.n - 1, 2)]: complex(1 / 2),
                MH.alg.index[(0, 1)]: complex(1 / 2 * omega ** (-k / 2)),
            }
        elif v == 2:
            mult_term_coeffs = {
                MH.alg.index[(MH.alg.n - 1, 2)]: complex(1 / 2),
                MH.alg.index[(0, 1)]: complex(-1 / 2 * omega ** (-k / 2)),
            }
        else:
            mult_term_coeffs = {}

        result_coeffs: Dict[int, complex] = {}
        for p_idx, p_coeff in pk_coeffs.items():
            for mult_idx, mult_coeff in mult_term_coeffs.items():
                product_of_basis_elements = MH.alg.multiply_basis(p_idx, mult_idx)
                for res_idx, res_coeff in product_of_basis_elements.items():
                    current_val = result_coeffs.get(res_idx, complex(0))
                    result_coeffs[res_idx] = current_val + p_coeff * mult_coeff * res_coeff

        return {
            idx: coeff
            for idx, coeff in result_coeffs.items()
            if abs(coeff) > MH.tolerance
        }

    # Compute basis change numerically
    basis_change_np = np.zeros((3 * n, 3 * n), dtype=complex)
    for i in range(3):
        for j in range(n):
            for idx, coeff in E_k(j, i).items():
                basis_change_np[i * n + j, idx] = coeff

    basis_change_np = basis_change_np.T
    basis_change_inv_np = np.linalg.inv(basis_change_np)

    tau_a_np = np.zeros((3 * n, 3 * n), dtype=complex)
    for i in range(3 * n):
        tau_a_np[i, i] = 1.0

    for a in A:
        tau_a_np[n + a, n + a] = 0.0
        tau_a_np[2 * n + a, 2 * n + a] = 0.0
        tau_a_np[2 * n + a, n + a] = 1.0
        tau_a_np[n + a, 2 * n + a] = 1.0

    # Compute prod and tensor product numerically via Kronecker product (np.kron)
    prod_np = basis_change_np @ tau_a_np @ basis_change_inv_np

    # Determine number of circles k in the starting state (homological degree 0)
    start_state = (0,) * len(MH.matching)
    k_circles = MH.smoothing_data(start_state).k

    prod_tensor_np = prod_np
    for _ in range(k_circles - 1):
        prod_tensor_np = np.kron(prod_tensor_np, prod_np)

    # Efficient quadratic form trace using fast NumPy vectorized operations
    trace_val = 0.0
    for i in range(null_len):
        v_i = nullspace_vectors_np[i]
        trace_val += (v_i.conj().T @ prod_tensor_np @ v_i)[0, 0]

    # Simplify/round result to nearest integer since colorings must be integer-valued
    return round(trace_val.real)

You can now run the code below to calcuate numbers of face colorings with your given choice of ribbon graph and set A.

In [8]:
# Replace the cycles input with your graph cycle data
cycles = [[[1, 2], [1, 1], [1, 0]], [[0, 2], [0, 1], [0, 0]]]

# Choice of parameter n
n = 1

# Choice of set A
A = []

color_num = 3*n - 2*len(A)
trace = get_face_color(cycles, n, A)
print(f"Number of {color_num}-face colorings: {int(trace)}")

Number of 3-face colorings: 6
